# 第 20 章互动实验：合成缺陷与扩散前向过程

比较“先插缺陷再成像”和“成像后直接贴缺陷”的差别，并可视化 DDPM 的前向加噪。

In [ ]:
%matplotlib inline
import importlib.util, os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["PingFang SC", "Heiti SC", "Noto Sans CJK SC", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

CHAPTER_NAME = "20_合成缺陷与生成模型"
ASSIGNMENT = "20-1_缺陷合成与扩散前向.py"

def find_chapter(name):
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if p.name == name:
            return p
        if (p / name).is_dir():
            return p / name
    raise FileNotFoundError("请在课程目录内启动 Jupyter")

CHAPTER = find_chapter(CHAPTER_NAME)
COURSE_ROOT = CHAPTER.parent

def load_module(chapter_dir, filename):
    override = os.environ.get("COURSE_IMPL_DIR")
    base = Path(override) / chapter_dir.name if override else chapter_dir
    spec = importlib.util.spec_from_file_location(Path(filename).stem, base / filename)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

mod = load_module(CHAPTER, ASSIGNMENT)

def attempt(fn, *args, **kwargs):
    """调用作业函数；还没实现时打印提示并返回 None，而不是中断整个 Notebook。"""
    try:
        return fn(*args, **kwargs)
    except NotImplementedError as exc:
        print(f"先完成作业里的 {fn.__name__}：{exc}")
        return None

try:
    import cv2  # 可选，用于和 OpenCV 对照
except ImportError:
    cv2 = None
print("作业文件：", CHAPTER / ASSIGNMENT, "｜OpenCV：", "可用" if cv2 else "未安装（对照实验会跳过）")


In [ ]:
def blur(img, sigma):
    fy = np.fft.fftfreq(img.shape[0])[:, None]; fx = np.fft.fftfreq(img.shape[1])[None, :]
    return np.real(np.fft.ifft2(np.fft.fft2(img) * np.exp(-2 * (np.pi * sigma) ** 2 * (fy**2 + fx**2))))

rng = np.random.default_rng(0)
y, x = np.mgrid[0:96, 0:96]
clean = 0.3 + 0.4 * ((x % 10) < 5)
m = attempt(mod.scratch_mask, (96, 96), (20.0, 15.0), (70.0, 80.0), 2.0, 1.0)
if m is not None:
    right = blur(mod.blend(clean, 0.95, m), 1.5) + rng.normal(scale=0.03, size=clean.shape)
    wrong = mod.blend(blur(clean, 1.5) + rng.normal(scale=0.03, size=clean.shape), 0.95, m)
    fig, ax = plt.subplots(1, 3, figsize=(12, 4))
    ax[0].imshow(m, cmap="gray"); ax[0].set_title("软掩码（标签）")
    ax[1].imshow(right, cmap="gray"); ax[1].set_title("先插缺陷再模糊加噪（正确）")
    ax[2].imshow(wrong, cmap="gray"); ax[2].set_title("成像后直接贴（贴纸感）")
    plt.show()
    print("观察：右图划痕边缘比背景锐利、没有噪声。检测器很容易学到这种捷径。")

## DDPM 前向过程

预测：t=300 时图像还能认出条纹吗？

In [ ]:
betas = attempt(mod.linear_beta_schedule, 1000)
if betas is not None:
    abar = mod.alpha_bars(betas)
    x0 = (clean - 0.5) * 2
    noise = np.random.default_rng(1).normal(size=x0.shape)
    ts = [0, 50, 150, 300, 600, 999]
    fig, ax = plt.subplots(1, len(ts), figsize=(15, 3))
    for a, t in zip(ax, ts):
        a.imshow(mod.q_sample(x0, t, abar, noise), cmap="gray"); a.set_title(f"t={t}, ᾱ={abar[t]:.3f}"); a.axis("off")
    plt.show()
    plt.plot(np.sqrt(abar), label="信号系数 sqrt(ᾱ)"); plt.plot(np.sqrt(1 - abar), label="噪声系数 sqrt(1-ᾱ)")
    plt.xlabel("t"); plt.legend(); plt.show()

## VAE 重参数化：采样的梯度从哪里来

In [ ]:
z = attempt(mod.reparameterize, np.zeros(5), np.log(np.full(5, 4.0)), np.ones(5))
if z is not None:
    print("mu=0、方差 4、eps=1 时 z =", z, "（dz/dmu = 1，dz/dlogvar = 0.5·sigma·eps，都和 eps 的采样无关）")

## 破坏实验与记录

- 把软掩码的 softness 设得很小（接近硬边），再看“贴纸感”。
- 用第 24 章的 `数据生成.py`，只用划痕缺陷调阈值，再测颗粒和缺失，检出率差多少？
- 记录到 `实验记录.md`。